# Milestone 2: Data Audit, Baseline, and Evaluation Design
**Predicting Diabetes Risk from Health Indicators and Lifestyle Factors**  
CAPS-DATASTUDIES 5040, Applied Machine Learning, Fall 2026

Requirements checklist: `MILESTONE2_REQUIREMENTS.md`. Data dictionary and scope: `README.md`.

**Order of operations.** The final test set is split off and locked in Section 0.3, *before* any EDA or fitting. Sections 3 to 8 use only the development set.

| Section | Topic | Pts |
|---|---|---|
| 1 | Project continuity and feedback matrix | 10 |
| 2 | Provenance and structural audit | 10 |
| 3 | Data quality and missingness audit | 12 |
| 4 | Exploratory data analysis | 8 |
| 5 | Leakage and feature availability | 10 |
| 6 | Preprocessing strategy | 10 |
| 7 | Baseline model | 10 |
| 8 | Evaluation metrics | 10 |
| 9 | Validation design and generalization | 15 |
| 10 | Workflow Canvas v2 and reproducibility | 5 |
| 11 | Final reflection | - |

## 0. Setup, reproducibility, and data access

### 0.1 Imports, seed, versions

In [ ]:
import sys, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (make_scorer, precision_score, recall_score, average_precision_score,
                             roc_auc_score, log_loss, brier_score_loss,
                             confusion_matrix, ConfusionMatrixDisplay,
                             PrecisionRecallDisplay)
from sklearn.calibration import CalibrationDisplay

SEED = 5040
np.random.seed(SEED)
pd.set_option("display.max_columns", 50)

print("Python      :", sys.version.split()[0], "|", platform.platform())
for m in (np, pd, sklearn):
    print(f"{m.__name__:<12}:", m.__version__)

### 0.2 Load the data
Install once if needed: `pip install ucimlrepo`. The raw download is cached to `data/` (git-ignored) so reruns do not need the network. Do not commit the data.

In [ ]:
from pathlib import Path
DATA = Path("data"); DATA.mkdir(exist_ok=True)
CACHE = DATA / "cdc_diabetes_raw.csv"

if CACHE.exists():
    df = pd.read_csv(CACHE)
else:
    from ucimlrepo import fetch_ucirepo
    ds = fetch_ucirepo(id=891)
    df = pd.concat([ds.data.ids, ds.data.features, ds.data.targets], axis=1)
    df.to_csv(CACHE, index=False)

print(df.shape)
df.head()

### 0.3 Lock the final test set (before any EDA)
Procedure and justification are in Section 9. Exact duplicate rows are removed first so identical respondents cannot land on both sides of the split (see Section 3 for the duplicate audit). Caveat: duplicates are matched on features *and* label, so feature profiles that appear with conflicting labels (counted in 3.2) can still straddle the split; discuss this in 3.2.  
The test set is **not used again** until the final evaluation in a later milestone.

In [ ]:
TARGET = "Diabetes_binary"
ID_COL = "ID" if "ID" in df.columns else None

feature_cols = [c for c in df.columns if c not in (TARGET, ID_COL)]
df_unique = df.drop_duplicates(subset=feature_cols + [TARGET]).reset_index(drop=True)

dev, test = train_test_split(df_unique, test_size=0.20, stratify=df_unique[TARGET],
                             random_state=SEED)
dev = dev.reset_index(drop=True)
test = test.reset_index(drop=True)   # locked: do not inspect or use for decisions

X_dev, y_dev = dev[feature_cols], dev[TARGET]
print(f"raw {len(df):,} -> deduplicated {len(df_unique):,} -> dev {len(dev):,} / test {len(test):,}")

## 1. Project continuity and Milestone 1 feedback (10 pts)

### 1.1 Restatement
- **Analytical question:** TODO
- **Stakeholder and decision:** TODO (public health programs / primary care; flag for follow-up testing, not diagnosis)
- **Observation:** TODO (one BRFSS respondent)
- **Target:** TODO (reported diabetes status at survey time; state exactly which classes are in y = 1 after Section 2)
- **Use time:** TODO
- **Generalization objective:** TODO (new respondents from the same survey wave; what is *not* claimed)

### 1.2 Mathematical contract
TODO: define with LaTeX, e.g.

$\mathcal{Z}=\mathcal{X}\times\mathcal{Y}$, $Z=(X,Y)$, $z_i=(x_i,y_i)$, $S_n=\{z_i\}_{i=1}^n$

- index $i$: TODO
- $x_i \in \mathcal{X}$: TODO (feature vector, 21 survey answers)
- $y_i \in \mathcal{Y}=\{0,1\}$: TODO
- group / time / location index: TODO (none, or justify)
- Distinguish random $Z$, realized $z$, finite sample $S_n$.

### 1.3 Feedback response matrix
Fill `Evidence` with section/figure references and `Status` with Resolved / Partially resolved / Open after the analysis. Add a next step for every unresolved row.

In [ ]:
feedback = pd.DataFrame([
 ("Clarify diagnosed vs. undiagnosed vs. future risk", "State target as reported diagnosis at survey time", "Sec. 1, 2", "TODO"),
 ("Justify combining prediabetes and diabetes", "Check what Diabetes_binary=1 really contains; decide binary vs. multiclass", "Sec. 2, 3", "TODO"),
 ("Formalize z_i = (x_i, y_i)", "Notation block", "Sec. 1.2", "TODO"),
 ("Feature definitions, coding, availability; HighBP/HighChol reflect prior evaluation or access", "Dictionary + availability table", "Sec. 2, 5", "TODO"),
 ("Examine variable definitions and coding", "Verify codes and ranges with value counts", "Sec. 2, 3", "TODO"),
 ("Interpret risks; no causal claims", "Association-vs-causation notes in EDA", "Sec. 4", "TODO"),
 ("Random split does not show contemporary generalization", "Define 'unseen'; list what is not shown", "Sec. 9", "TODO"),
 ("BRFSS complex sampling / survey weights", "Check for weight columns; state consequence", "Sec. 2, 9", "TODO"),
 ("Define learned preprocessing; prevent leakage", "Pipeline, fit on train folds only", "Sec. 6, 9", "TODO"),
 ("Expand Canvas: baseline, methods, preprocessing, metrics", "Canvas v2", "Sec. 10", "TODO"),
 ("Error costs: missed case vs. unnecessary follow-up", "Recall / precision / PR-AUC, threshold discussion", "Sec. 8", "TODO"),
], columns=["Instructor feedback", "Action or justified decision", "Evidence in Milestone 2", "Status"])
feedback

**Open items and next steps:** TODO

## 2. Data provenance and structural audit (10 pts)

### 2.1 Provenance
TODO: source (UCI id 891, DOI 10.24432/C53919; CDC BRFSS), collection method (telephone survey, self-report), survey year (**resolve 2014 vs. 2015**: UCI text says 2014, Kaggle file names say 2015), labeling process, license, sensitive fields.

### 2.2 Dataset inventory (full raw data, no decisions made from it)

In [ ]:
inventory = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_unique": df.nunique(),
    "min": df.min(numeric_only=True),
    "max": df.max(numeric_only=True),
})
print(f"rows={len(df):,}  cols={df.shape[1]}  memory={df.memory_usage(deep=True).sum()/1e6:.1f} MB")
inventory

### 2.3 Data dictionary
TODO: copy/extend from `README.md` (variable, meaning, type, source, measurement/availability time, coding notes). Cross-check codes against the inventory above.

In [ ]:
data_dictionary = pd.DataFrame([
 # variable, meaning, type, availability, coding notes
 ("Diabetes_binary", "Reported diabetes status", "binary (target)", "survey time", "0 = no; 1 = prediabetes or diabetes per UCI (VERIFY)"),
 ("HighBP", "Told high blood pressure", "binary", "TODO", "TODO"),
 ("HighChol", "Told high cholesterol", "binary", "TODO", "TODO"),
 # TODO: add remaining 19 features
], columns=["variable", "meaning", "type", "availability_time", "coding_notes"])
data_dictionary

### 2.4 Structural checks
Confirm: a row is one respondent; identifier column; no group/time/spatial column; no survey weight / strata / PSU column; single table (no joins, so no keys or unmatched records).

**Target contents check.** Compare the positive count to the three-class Kaggle file (`Diabetes_012`: 0 none, 1 prediabetes, 2 diabetes) to learn whether prediabetes is in class 0 or class 1. Record the result and its consequence for Section 1.

In [ ]:
print(df[TARGET].value_counts().sort_index())
print("Columns that could be weights/strata/ids:", [c for c in df.columns if any(k in c.lower() for k in ("wt", "weight", "strat", "psu", "id"))])
# TODO: download/load the 3-class file and compare counts (state the source and date you retrieved it)

## 3. Data quality and missingness audit (12 pts)
Descriptive only. Any fitted imputer belongs inside the training fold (Section 6).

$\hat m_j=\frac1n\sum_{i=1}^n \mathbb 1\{x_{ij}\text{ is missing}\}$

### 3.1 Missingness

In [ ]:
miss = (df.isna().mean().rename("missing_share").to_frame()
          .assign(missing_count=df.isna().sum()))
miss[miss.missing_count > 0] if miss.missing_count.any() else print("No NaN values in any column.")

TODO: also check for sentinel codes that look like valid values (e.g. 7, 9, 77, 99 for don't know / refused) in the ordinal and day-count columns.

### 3.2 Duplicates

In [ ]:
dup_mask = df.duplicated(subset=feature_cols + [TARGET], keep=False)
n_dup_rows = df.duplicated(subset=feature_cols + [TARGET]).sum()
conflicts = (df.groupby(feature_cols)[TARGET].nunique() > 1).sum()
print(f"redundant duplicate rows: {n_dup_rows:,} ({n_dup_rows/len(df):.1%})")
print(f"feature profiles with conflicting labels: {conflicts:,}")

TODO: interpret. Identical answer profiles are plausible in a 21-variable, mostly binary survey, so duplicates are not necessarily errors. State why dropping them before the split is still the safer choice, and its effect on prevalence.

### 3.3 Invalid values and ranges

In [ ]:
expected = {  # column: (min, max)  -- from the data dictionary
    "Age": (1, 13), "GenHlth": (1, 5), "Education": (1, 6), "Income": (1, 8),
    "MentHlth": (0, 30), "PhysHlth": (0, 30),
}
binary_cols = [c for c in feature_cols if df[c].nunique() <= 2 and c not in expected]
rows = []
for c, (lo, hi) in expected.items():
    rows.append((c, lo, hi, int(((df[c] < lo) | (df[c] > hi)).sum())))
for c in binary_cols:
    rows.append((c, 0, 1, int((~df[c].isin([0, 1])).sum())))
pd.DataFrame(rows, columns=["column", "min_ok", "max_ok", "n_out_of_range"])

### 3.4 Outliers (BMI)

In [ ]:
print(dev["BMI"].describe())
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
dev["BMI"].plot.hist(bins=60, ax=ax[0], title="BMI (dev)")
dev.boxplot("BMI", by=TARGET, ax=ax[1]); ax[1].set_title("BMI by target")
plt.suptitle(""); plt.tight_layout(); plt.show()

### 3.5 Class balance

In [ ]:
bal = pd.concat({"full (raw)": df[TARGET].value_counts(),
                 "dev": dev[TARGET].value_counts(),
                 "test": test[TARGET].value_counts()}, axis=1)
bal.loc["prevalence of y=1"] = [d[TARGET].mean() for d in (df, dev, test)]
bal

**Interpretation, exclusions, treatments, limitations:** TODO  
Note label availability (all rows labeled), group/time/source coverage (single wave), and the consequence of imbalance for metric choice.

## 4. Exploratory data analysis (8 pts)
**Development set only.** Each analysis: question, finding, implication, and an association-not-causation note.

### 4.1 Class distribution
**Question:** TODO

In [ ]:
y_dev.value_counts(normalize=True).plot.bar(title="Target distribution (dev)")
plt.ylabel("share"); plt.show()

**Finding:** TODO  
**Implication:** TODO  
**Association vs. causation:** TODO

### 4.2 Target rate by an ordered predictor (Age, GenHlth, Income, BMI)
**Question:** TODO

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, c in zip(axes, ["Age", "GenHlth", "Income"]):
    dev.groupby(c)[TARGET].mean().plot(marker="o", ax=ax, title=f"P(y=1) by {c}")
    ax.set_ylabel("rate")
plt.tight_layout(); plt.show()

**Finding:** TODO  
**Implication:** TODO  
**Association vs. causation:** TODO

### 4.3 Binary predictors versus the target (comorbidity / access features)
**Question:** TODO (do `HighBP`, `HighChol`, `DiffWalk`, `CholCheck` act as legitimate predictors or as near-proxies of the outcome?)

In [ ]:
bin_feats = [c for c in feature_cols if dev[c].nunique() == 2]
rate = pd.DataFrame({c: dev.groupby(c)[TARGET].mean() for c in bin_feats}).T
rate.columns = ["P(y=1 | x=0)", "P(y=1 | x=1)"]
rate["risk_ratio"] = rate.iloc[:, 1] / rate.iloc[:, 0]
rate.sort_values("risk_ratio", ascending=False)

**Finding:** TODO  
**Implication:** TODO  
**Association vs. causation:** TODO

### 4.4 (Optional) Feature-feature correlation

In [ ]:
corr = X_dev.corr()
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
plt.colorbar(im); plt.tight_layout(); plt.show()

## 5. Leakage and feature availability assessment (10 pts)
**Information boundary (non-temporal task):** TODO. Define what a screener would know at intake, and what is a consequence of already having a diabetes diagnosis (e.g. diagnosis leads to blood-pressure/cholesterol checks, mobility or health-status changes).

For every operation ask: what information does it observe, what state does it learn, and would it be available at prediction time?

In [ ]:
leakage = pd.DataFrame([
 # feature or group, availability at use time, leakage / dependency risk, decision / evidence
 ("ID", "n/a", "Row identifier; no signal, could encode file order", "Exclude"),
 ("HighBP, HighChol", "TODO", "May reflect prior medical evaluation or access; comorbid with diabetes", "Investigate"),
 ("CholCheck, AnyHealthcare, NoDocbcCost", "TODO", "Healthcare contact / access proxies", "Investigate"),
 ("GenHlth, PhysHlth, MentHlth, DiffWalk", "TODO", "May be downstream of diabetes", "Investigate"),
 ("Stroke, HeartDiseaseorAttack", "TODO", "Comorbidities / complications", "Investigate"),
 ("BMI, Smoker, PhysActivity, Fruits, Veggies, HvyAlcoholConsump", "TODO", "Self-report error; none identified", "Retain"),
 ("Age, Sex, Education, Income", "TODO", "Sensitive attributes; none identified for leakage", "Retain"),
 ("Duplicates across split", "n/a", "Identical rows on both sides inflate scores", "Deduplicated before split (Sec. 0.3)"),
 ("Fitted preprocessing", "n/a", "Scaler/imputer fit on all data leaks validation info", "Fit inside pipeline per fold"),
], columns=["feature / group", "availability at use time", "leakage or dependency risk", "decision / evidence"])
leakage

**Still needs verification:** TODO (list assumptions that cannot be checked from this file, such as exact question wording and question order in the BRFSS codebook).

## 6. Preprocessing and feature preparation (10 pts)

### 6.1 Deterministic vs. fitted operations
| Operation | Type | State learned | Fitting set |
|---|---|---|---|
| Drop `ID`, drop exact duplicates | deterministic | none | n/a |
| TODO: range/code cleaning | deterministic | none | n/a |
| Standardize continuous columns | **fitted** | $\hat\mu_j,\hat\sigma_j$ | training fold only |
| TODO: imputation / encoding | fitted | TODO | training fold only |

### 6.2 Formal statement
$\hat\eta = \mathrm{Fit}_T(S_{tr}),\quad x' = T_{\hat\eta}(x)$

$\hat\mu_j=\frac1{n_{tr}}\sum_{i\in\mathcal I_{tr}}x_{ij},\quad \hat\sigma_j=\sqrt{\frac1{n_{tr}}\sum_{i\in\mathcal I_{tr}}(x_{ij}-\hat\mu_j)^2},\quad x'_{ij}=\frac{x_{ij}-\hat\mu_j}{\hat\sigma_j}$

Zero-variance rule: TODO (scikit-learn sets scale to 1 when $\hat\sigma_j=0$; state this). Validation/test inputs use the fitted $T_{\hat\eta}$ with no refitting.

TODO: decide whether ordinal columns (Age, GenHlth, Education, Income) are treated as numeric or categorical, and justify.

In [ ]:
continuous = ["BMI", "MentHlth", "PhysHlth"]
ordinal    = ["Age", "GenHlth", "Education", "Income"]       # TODO: numeric vs. one-hot
passthrough = [c for c in feature_cols if c not in continuous]

preprocess = ColumnTransformer(
    [("scale", StandardScaler(), continuous + ordinal)],     # TODO: revisit ordinal handling
    remainder="passthrough",
)
preprocess

## 9. Validation design and generalization (15 pts)
*(Placed before the baseline because the baseline's evaluation depends on it.)*

### 9.1 What "unseen" means
- **Claim tested:** TODO (performance on new respondents from the same survey wave, assuming rows are roughly independent).
- **Not demonstrated:** TODO (other years, contemporary clinical populations, undiagnosed or future cases, survey-weighted population estimates, subgroup fairness).

### 9.2 Partition procedure
$\mathcal D=\mathcal D_{train}\cup\mathcal D_{val}\cup\mathcal D_{test},\quad \mathcal D_a\cap\mathcal D_b=\emptyset\ (a\ne b)$

Stratified random, because no group/time index exists. TODO: justify; note row-disjointness is not independence if respondents repeat (no ID evidence of that here). Test set locked in Section 0.3; development set evaluated by stratified K-fold.

In [ ]:
N_SPLITS = 5
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

sizes = pd.DataFrame({"n": [len(dev), len(test)],
                      "prevalence y=1": [y_dev.mean(), test[TARGET].mean()]},
                     index=["development", "test (locked)"])
display(sizes)

fold_rows = []
for k, (tr, va) in enumerate(cv.split(X_dev, y_dev), 1):
    fold_rows.append((k, len(tr), len(va), y_dev.iloc[tr].mean(), y_dev.iloc[va].mean()))
pd.DataFrame(fold_rows, columns=["fold", "n_train", "n_val", "prev_train", "prev_val"])

### 9.3 Fold contract
In every fold $k$: $S_{tr,k}\to(\hat\eta_k,\hat\theta_k)$ and $S_{val,k}\to\mathrm{Eval}(\hat M_k)$. No fitted state transfers across folds; the full `Pipeline` is cloned and refit per fold by `cross_validate`.

### 9.4 Survey design
TODO: BRFSS uses a complex sampling design (stratified, weighted). This file has no weight/strata/PSU columns, so estimates describe the *sample*, not the US adult population. State how that limits the claim.

## 7. Baseline model and initial results (10 pts)

$\hat\theta=\mathrm{Fit}_f(\{(T_{\hat\eta}(x_i),y_i):i\in\mathcal I_{tr}\}),\quad \hat M(x)=f_{\hat\theta}(T_{\hat\eta}(x)),\quad \hat y=\hat M(x)$

- **Reference 1:** `DummyClassifier` (floor).
- **Reference 2:** logistic regression (matches the project's stated model; interpretable).
- **State learned:** preprocessing $\hat\eta$ = scaler means/scales. Model $\hat\theta$ = coefficients, intercept.
- **Assumptions:** TODO (independent rows, linear log-odds, no survey weights).
- **Output type:** probabilities, thresholded to labels at a stated cut-off.
- **Imbalance handling:** compare `class_weight=None` vs `"balanced"`; no extensive tuning.

In [ ]:
models = {
    "dummy (prior)": Pipeline([("prep", preprocess),
                               ("clf", DummyClassifier(strategy="prior"))]),
    "logreg": Pipeline([("prep", preprocess),
                        ("clf", LogisticRegression(max_iter=1000))]),
    "logreg balanced": Pipeline([("prep", preprocess),
                                 ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))]),
}

## 8. Primary evaluation metrics (10 pts)

$\mathrm{Precision}=\frac{TP}{TP+FP}\qquad \mathrm{Recall}=\frac{TP}{TP+FN}$

- **Primary:** recall of y=1 at a stated threshold (a missed case is costly), with precision reported alongside (follow-up workload). TODO: justify.
- **Ranking:** PR-AUC (`average_precision`), threshold-free and sensitive to imbalance. ROC-AUC as secondary.
- **Probability quality:** log loss, Brier score, calibration curve. (Note `balanced` weighting distorts calibration.)
- **Why not accuracy:** TODO (always predicting 0 scores high accuracy).
- **Error consequences:** TODO (FN vs. FP in screening context). **Limitations:** TODO.

In [ ]:
scoring = {
    "recall": "recall",
    "precision": make_scorer(precision_score, zero_division=0),
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "neg_log_loss": "neg_log_loss",
    "neg_brier": "neg_brier_score",
}
results = {}
for name, pipe in models.items():
    cvres = cross_validate(pipe, X_dev, y_dev, cv=cv, scoring=scoring, n_jobs=-1)
    results[name] = {m: (cvres[f"test_{m}"].mean(), cvres[f"test_{m}"].std()) for m in scoring}

summary = pd.DataFrame({n: {m: f"{mu:.3f} ± {sd:.3f}" for m, (mu, sd) in r.items()}
                        for n, r in results.items()}).T
summary

**Interpretation:** TODO (what the dummy shows about imbalance; gain of logistic over dummy; recall/precision trade-off at the default 0.5 threshold; effect of class weighting).

### Out-of-fold predictions: confusion matrix, PR curve, calibration

In [ ]:
from sklearn.model_selection import cross_val_predict
pipe = models["logreg"]
proba = cross_val_predict(pipe, X_dev, y_dev, cv=cv, method="predict_proba")[:, 1]
THRESH = 0.5   # TODO: justify; compare other thresholds via the PR curve

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ConfusionMatrixDisplay(confusion_matrix(y_dev, proba >= THRESH)).plot(ax=ax[0], colorbar=False)
ax[0].set_title(f"Confusion matrix @ {THRESH}")
PrecisionRecallDisplay.from_predictions(y_dev, proba, ax=ax[1]); ax[1].set_title("PR curve (OOF)")
CalibrationDisplay.from_predictions(y_dev, proba, n_bins=10, ax=ax[2]); ax[2].set_title("Calibration (OOF)")
plt.tight_layout(); plt.show()

**Coefficients (fit on full dev set, for interpretation only; associations, not causal effects):**

In [ ]:
fitted = models["logreg"].fit(X_dev, y_dev)
names = fitted.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(fitted.named_steps["clf"].coef_[0], index=names).sort_values()
coefs

## 10. Workflow Canvas v2 and reproducibility (5 pts)

### 10.1 Canvas v2
Chain: decision/problem → observation → measurement and features → preparation → baseline learning → model output → validation → metric(s) → intended use.  
Composition: $M=M_K\circ\cdots\circ M_1$.

| Stage | Content | Fixed or learned | Stored state | Fitting sample |
|---|---|---|---|---|
| Decision / problem | TODO | n/a | n/a | n/a |
| Observation | One BRFSS respondent | fixed | n/a | n/a |
| Measurement and features | 21 self-reported variables | fixed | n/a | n/a |
| Preparation | drop ID/duplicates (fixed); standardize (learned) | mixed | $\hat\mu,\hat\sigma$ | training fold |
| Baseline learning | dummy; logistic regression | learned | $\hat\theta$ | training fold |
| Model output | probability of y=1, thresholded | n/a | threshold | TODO |
| Validation | stratified 5-fold on dev; test locked | n/a | n/a | n/a |
| Metrics | recall, precision, PR-AUC, calibration | n/a | n/a | n/a |
| Intended use | prioritize follow-up testing | n/a | n/a | n/a |

**Claim tested by validation:** TODO  
**Risks and evidence from this audit:** TODO  
**Outstanding decisions:** TODO (binary vs. multiclass target; ordinal handling; feature exclusions from Section 5; threshold)

### 10.2 How to rerun
1. `pip install -r requirements.txt` (TODO: create; or `pip install numpy pandas scikit-learn matplotlib ucimlrepo`)
2. Open the notebook, *Restart kernel and Run All*. The first run downloads the data via `ucimlrepo` and caches it in `data/` (git-ignored).
3. All randomness uses `SEED` (Section 0.1).

Versions are printed in Section 0.1.

## 11. Final reflection
- **What changed after Milestone 1 feedback:** TODO
- **What the data audit revealed:** TODO
- **Assumptions challenged:** TODO
- **What the baseline and validation show and do not show:** TODO
- **Next evidence needed:** TODO